In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import numpy as np

from pytorch_metric_learning.losses import MultiSimilarityLoss
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    accuracy_score, precision_score,
    recall_score, f1_score
)

# -------------------------------
# CONFIG
# -------------------------------
ROOT = "/kaggle/input/datasets/narendra0409/cattle-1-verification3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 48
EPOCHS = 50
LR = 1e-5
EMB_DIM = 512

MODEL_SAVE_PATH = "stage1_best_by_val_loss.pth"

# -------------------------------
# TRANSFORMS
# -------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# -------------------------------
# DATASET
# -------------------------------

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

class CattleStage1Dataset(Dataset):
    def __init__(self, root, split):
        self.samples = []
        base = os.path.join(root, split)

        for cid in sorted(os.listdir(base)):
            fdir = os.path.join(base, cid, "face")
            mdir = os.path.join(base, cid, "muzzle")

            if not os.path.isdir(fdir) or not os.path.isdir(mdir):
                continue

            face_imgs = [
                f for f in sorted(os.listdir(fdir))
                if f.lower().endswith(VALID_EXTENSIONS)
            ]

            muzzle_imgs = [
                m for m in sorted(os.listdir(mdir))
                if m.lower().endswith(VALID_EXTENSIONS)
            ]

            for f, m in zip(face_imgs, muzzle_imgs):
                self.samples.append((
                    os.path.join(fdir, f),
                    os.path.join(mdir, m),
                    int(cid)
                ))
# class CattleStage1Dataset(Dataset):
#     def __init__(self, root, split):
#         self.samples = []
#         base = os.path.join(root, split)

#         for cid in sorted(os.listdir(base)):
#             fdir = os.path.join(base, cid, "face")
#             mdir = os.path.join(base, cid, "muzzle")

#             if not os.path.isdir(fdir) or not os.path.isdir(mdir):
#                 continue

#             for f, m in zip(sorted(os.listdir(fdir)), sorted(os.listdir(mdir))):
#                 self.samples.append((
#                     os.path.join(fdir, f),
#                     os.path.join(mdir, m),
#                     int(cid)
#                 ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f, m, y = self.samples[idx]
        face = transform(Image.open(f).convert("RGB"))
        muzzle = transform(Image.open(m).convert("RGB"))
        return face, muzzle, y

# -------------------------------
# MODEL
# -------------------------------
class Stage1Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )
        self.fc = nn.Linear(self.backbone.num_features, EMB_DIM)

    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return F.normalize(x, dim=1)

# -------------------------------
# LOADERS
# -------------------------------
train_loader = DataLoader(
    CattleStage1Dataset(ROOT, "train"),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    CattleStage1Dataset(ROOT, "val"),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# -------------------------------
# SETUP
# -------------------------------
model = Stage1Model().to(DEVICE)

criterion = MultiSimilarityLoss(alpha=2.0, beta=40.0, base=0.5)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-6
)

# -------------------------------
# TRAIN + VAL LOOP
# -------------------------------
print("\n===== STAGE-1 TRAINING =====")

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # -------- TRAIN --------
    model.train()
    train_loss = 0.0
    train_scores = []

    for face, muzzle, label in train_loader:
        face, muzzle, label = face.to(DEVICE), muzzle.to(DEVICE), label.to(DEVICE)

        f_emb = model(face)
        m_emb = model(muzzle)

        emb = torch.cat([f_emb, m_emb], dim=0)
        lab = torch.cat([label, label], dim=0)

        loss = criterion(emb, lab)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_scores.extend((f_emb * m_emb).sum(dim=1).detach().cpu().numpy())

    train_acc = (np.array(train_scores) >= 0).mean() * 100

    # -------- VAL --------
    model.eval()
    val_loss = 0.0
    val_scores = []

    with torch.no_grad():
        for face, muzzle, label in val_loader:
            face, muzzle, label = face.to(DEVICE), muzzle.to(DEVICE), label.to(DEVICE)

            f_emb = model(face)
            m_emb = model(muzzle)

            emb = torch.cat([f_emb, m_emb], dim=0)
            lab = torch.cat([label, label], dim=0)

            loss = criterion(emb, lab)
            val_loss += loss.item()
            val_scores.extend((f_emb * m_emb).sum(dim=1).cpu().numpy())

    val_acc = (np.array(val_scores) >= 0).mean() * 100

    # -------- SAVE BEST --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"⭐ Best model saved (Val Loss = {val_loss:.4f})")

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss={train_loss/len(train_loader):.4f} | "
        f"Train Acc={train_acc:.2f}% | "
        f"Val Loss={val_loss/len(val_loader):.4f} | "
        f"Val Acc={val_acc:.2f}%"
    )

print("\n✅ Training Finished. Best model saved.")

# ============================================================
# EVALUATION USING BEST MODEL
# ============================================================

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

@torch.no_grad()
def extract_embeddings(split, modality):
    feats, labels = [], []
    base = os.path.join(ROOT, split)

    for cid in sorted(os.listdir(base)):
        mdir = os.path.join(base, cid, modality)
        if not os.path.isdir(mdir):
            continue

        for img in sorted(os.listdir(mdir)):
            im = transform(Image.open(os.path.join(mdir, img)).convert("RGB"))
            emb = model(im.unsqueeze(0).to(DEVICE))
            feats.append(emb.cpu().numpy()[0])
            labels.append(int(cid))

    return np.array(feats), np.array(labels)

def evaluate(face_feat, face_lab, muz_feat, muz_lab, neg_per_pos=1):
    scores, gt = [], []
    ids = np.unique(face_lab)

    for cid in ids:
        f_idx = np.where(face_lab == cid)[0]
        m_idx = np.where(muz_lab == cid)[0]

        for i in f_idx:
            for j in m_idx:
                scores.append(np.dot(face_feat[i], muz_feat[j]))
                gt.append(1)

                neg_ids = ids[ids != cid]
                for nid in np.random.choice(neg_ids, neg_per_pos, replace=False):
                    jn = np.random.choice(np.where(muz_lab == nid)[0])
                    scores.append(np.dot(face_feat[i], muz_feat[jn]))
                    gt.append(0)

    scores, gt = np.array(scores), np.array(gt)

    fpr, tpr, th = roc_curve(gt, scores)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    preds = (scores >= th[idx]).astype(int)

    return {
        "EER": fpr[idx]*100,
        "TAR@1%": tpr[np.where(fpr <= 0.01)[0][-1]]*100,
        "ACC": accuracy_score(gt, preds)*100,
        "PREC": precision_score(gt, preds)*100,
        "RECALL": recall_score(gt, preds)*100,
        "F1": f1_score(gt, preds)*100,
        "AUC": roc_auc_score(gt, scores)*100
    }

face_feat, face_lab = extract_embeddings("test", "face")
muz_feat, muz_lab = extract_embeddings("test", "muzzle")

res = evaluate(face_feat, face_lab, muz_feat, muz_lab)

print("\n===== STAGE-1 TEST RESULTS (BEST MODEL) =====")
for k, v in res.items():
    print(f"{k}: {v:.2f}")


In [ ]:
# ============================================================
# RIDGEFORMER STAGE-2 (PAPER-FAITHFUL)
# Cross-Attention Refinement + Frozen Backbone
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import numpy as np

from pytorch_metric_learning.losses import MultiSimilarityLoss
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    accuracy_score, precision_score,
    recall_score, f1_score
)

# ---------------- CONFIG ----------------
ROOT = "/kaggle/input/datasets/narendra0409/cattle-1-verification3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
EPOCHS = 50
LR = 5e-6
EMB_DIM = 512

STAGE1_MODEL_PATH = "/kaggle/input/models/narendra0409/stage1-veri-cat-1/pytorch/default/1/stage1-veri-cat-1.pth"
STAGE2_SAVE_PATH = "best_stage2_model.pth"

valid_ext = (".jpg", ".jpeg", ".png", ".bmp")

# ---------------- TRANSFORM ----------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ---------------- DATASET ----------------
class Stage2Dataset(Dataset):
    def __init__(self, root, split):
        self.samples = []
        base = os.path.join(root, split)

        for cid in sorted(os.listdir(base)):
            fdir = os.path.join(base, cid, "face")
            mdir = os.path.join(base, cid, "muzzle")

            if not os.path.isdir(fdir) or not os.path.isdir(mdir):
                continue

            face_imgs = sorted([f for f in os.listdir(fdir) if f.lower().endswith(valid_ext)])
            muzzle_imgs = sorted([m for m in os.listdir(mdir) if m.lower().endswith(valid_ext)])

            min_len = min(len(face_imgs), len(muzzle_imgs))

            for i in range(min_len):
                self.samples.append((
                    os.path.join(fdir, face_imgs[i]),
                    os.path.join(mdir, muzzle_imgs[i]),
                    int(cid)
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f, m, y = self.samples[idx]

        try:
            face = transform(Image.open(f).convert("RGB"))
            muzzle = transform(Image.open(m).convert("RGB"))
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        return face, muzzle, y


# ---------------- MODEL ----------------
class CrossAttentionRefinement(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, f, m):
        f_q = f.unsqueeze(1)
        m_q = m.unsqueeze(1)

        f2m, _ = self.attn(f_q, m_q, m_q)
        m2f, _ = self.attn(m_q, f_q, f_q)

        fused = (f2m + m2f) / 2
        fused = self.norm(fused)
        fused = fused + self.ff(fused)
        fused = fused.squeeze(1)

        return F.normalize(fused, dim=1)


class Stage2Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=0
        )

        self.fc = nn.Linear(self.backbone.num_features, EMB_DIM)
        self.refine = CrossAttentionRefinement(EMB_DIM)

    def forward_once(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return F.normalize(x, dim=1)

    def forward(self, face, muzzle):
        f = self.forward_once(face)
        m = self.forward_once(muzzle)
        fused = self.refine(f, m)
        return f, m, fused


# ---------------- LOAD DATA ----------------
train_loader = DataLoader(Stage2Dataset(ROOT, "train"),
                          batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

val_loader = DataLoader(Stage2Dataset(ROOT, "val"),
                        batch_size=BATCH_SIZE, shuffle=False)

# ---------------- INIT MODEL ----------------
model = Stage2Model().to(DEVICE)

print("Loading Stage-1 weights...")

stage1 = torch.load(STAGE1_MODEL_PATH, map_location=DEVICE)

model.backbone.load_state_dict(
    {k.replace("backbone.", ""): v for k, v in stage1.items() if "backbone" in k},
    strict=False
)

model.fc.load_state_dict(
    {k.replace("fc.", ""): v for k, v in stage1.items() if "fc" in k},
    strict=False
)

for p in model.backbone.parameters():
    p.requires_grad = False

criterion = MultiSimilarityLoss(alpha=2.0, beta=40.0, base=0.5)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=LR, weight_decay=1e-6)

best_val_loss = float("inf")

# ============================================================
# TRAINING
# ============================================================
print("\n===== STAGE-2 TRAINING =====")

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for face, muzzle, label in train_loader:
        face, muzzle, label = face.to(DEVICE), muzzle.to(DEVICE), label.to(DEVICE)

        f, m, fused = model(face, muzzle)

        emb = torch.cat([f, m, fused], dim=0)
        lab = torch.cat([label, label, label], dim=0)

        loss = criterion(emb, lab)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for face, muzzle, label in val_loader:
            face, muzzle, label = face.to(DEVICE), muzzle.to(DEVICE), label.to(DEVICE)

            f, m, fused = model(face, muzzle)

            emb = torch.cat([f, m, fused], dim=0)
            lab = torch.cat([label, label, label], dim=0)

            loss = criterion(emb, lab)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train={train_loss:.4f} | Val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), STAGE2_SAVE_PATH)
        print("Best Stage-2 model saved!")


# ============================================================
# TESTING (FIXED)
# ============================================================

print("\nLoading best Stage-2 model for testing...")
model.load_state_dict(torch.load(STAGE2_SAVE_PATH))
model.eval()

np.random.seed(42)  # Deterministic evaluation


@torch.no_grad()
def extract_fused(split):
    feats, labels = [], []
    base = os.path.join(ROOT, split)

    for cid in sorted(os.listdir(base)):
        fdir = os.path.join(base, cid, "face")
        mdir = os.path.join(base, cid, "muzzle")

        if not os.path.isdir(fdir) or not os.path.isdir(mdir):
            continue

        face_imgs = sorted([f for f in os.listdir(fdir) if f.lower().endswith(valid_ext)])
        muzzle_imgs = sorted([m for m in os.listdir(mdir) if m.lower().endswith(valid_ext)])

        min_len = min(len(face_imgs), len(muzzle_imgs))

        for i in range(min_len):
            face = transform(Image.open(os.path.join(fdir, face_imgs[i])).convert("RGB")).unsqueeze(0).to(DEVICE)
            muzzle = transform(Image.open(os.path.join(mdir, muzzle_imgs[i])).convert("RGB")).unsqueeze(0).to(DEVICE)

            _, _, fused = model(face, muzzle)
            feats.append(fused.cpu().numpy()[0])
            labels.append(int(cid))

    return np.array(feats), np.array(labels)


def evaluate(feat, lab):
    scores, gt = [], []
    ids = np.unique(lab)

    for cid in ids:
        idx = np.where(lab == cid)[0]
        neg_ids = ids[ids != cid]

        # Genuine pairs (unique combinations only)
        for i in range(len(idx)):
            for j in range(i+1, len(idx)):
                scores.append(np.dot(feat[idx[i]], feat[idx[j]]))
                gt.append(1)

                # One deterministic negative
                nid = np.random.choice(neg_ids)
                jn = np.random.choice(np.where(lab == nid)[0])
                scores.append(np.dot(feat[idx[i]], feat[jn]))
                gt.append(0)

    scores, gt = np.array(scores), np.array(gt)

    fpr, tpr, th = roc_curve(gt, scores)
    fnr = 1 - tpr
    idx_eer = np.argmin(np.abs(fpr - fnr))
    preds = (scores >= th[idx_eer]).astype(int)

    return {
        "EER": fpr[idx_eer]*100,
        "TAR@1%": tpr[np.where(fpr <= 0.01)[0][-1]]*100 if np.any(fpr <= 0.01) else 0,
        "ACC": accuracy_score(gt, preds)*100,
        "PREC": precision_score(gt, preds)*100,
        "RECALL": recall_score(gt, preds)*100,
        "F1": f1_score(gt, preds)*100,
        "AUC": roc_auc_score(gt, scores)*100
    }


feat, lab = extract_fused("test")
res = evaluate(feat, lab)

print("\n===== STAGE-2 TEST RESULTS =====")
for k, v in res.items():
    print(f"{k}: {v:.2f}")